# Do CV-parser errors reach the selection? A fairness audit of the Adecco reverse matching

**AI Ethics, Module 2 – Hiring CV Bias** · Francesco Baiocchi, Leonardo Petrilli

## Phase 2 – Protocol

Phase 1 (`phase1_data_audit`) produced `data/ethics/pairs.parquet`: one row per (candidate, job) pair of the clean
candidates, without the score outliers, with Adecco's score and the score re-computed with the recovered formula on the
three profile versions P (Adecco's real profile), O (parser output) and C (parser output plus the recovered false negatives). Before any
fairness number is computed, this phase fixes **what counts as a selection** (2.1) and looks at the outcomes it gives
under the four scores (2.2).

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import polars as pl

warnings.filterwarnings("ignore")

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_float_precision(4)

polars.config.Config

### 2.1 The protocol, fixed before looking at fairness

Before computing any fairness number we decide what "selected" means.

* **Unit**: a (candidate, job) pair.
* **Selected**: a candidate is selected for a job if they are in the top half of that job's candidates, ranked by
  score. A score only means something compared with the other candidates of the same job, so the ranking is done
  inside each job, as in a direct matching. As a stricter second outcome we also use the top 3.
* **Ties**: when candidates of the same job have the same score, their order is drawn at random. Each pair gets one
  random number, and the same number is used with every score (Adecco, P, O, C). So every job selects exactly
  ⌈n/2⌉ candidates, and a pair can change outcome between two versions only because the scores change, never
  because of a different draw.
* **Jobs kept**: jobs with at least 5 candidates. In a shorter list, being in the top half says
  nothing. Candidates with gender `Other` or `Unknown` (117) are left out: they are too few for group statistics.
* **Scores**: Adecco's own score (`score_adecco`) and the score recomputed with our formula on the three profile
  versions (`score_P`, `score_O`, `score_C`).
* **Groups**: `Gender`, `Location`, and the two together (`group`, e.g. `Female_SOUTH`).
* **Confidence intervals** (Phase 3): we resample candidates, not pairs, because the same candidate appears in
  several pairs.

The file's `rev_match_rank` is the position of the job among the candidate's 10 recommendations, so it is not used:
the position among the job's candidates is computed from the score. Because the file holds only each candidate's top
10, a job's list contains the candidates for whom it was among the 10 best, not every candidate who applied: the
ranking is among those.

In [ ]:
from hiring_cv_bias.fairness.protocol import (
    SCORES,
    add_outcomes,
    filter_protocol,
    load_pairs,
    protocol_sizes,
)

pairs = load_pairs()
df = add_outcomes(filter_protocol(pairs), SCORES)
n_jobs, n_cand = pairs["Job_id"].n_unique(), pairs["CANDIDATE_ID"].n_unique()
print("kept by the protocol (jobs with at least 5 Female or Male candidates):")
print(
    f"  pairs:      {df.height:,} of {pairs.height:,} ({df.height / pairs.height:.1%})"
)
print(f"  jobs:       {df['Job_id'].n_unique():,} of {n_jobs:,}")
print(f"  candidates: {df['CANDIDATE_ID'].n_unique():,} of {n_cand:,}")
print(
    f"median number of candidates per job: {df.group_by('Job_id').len()['len'].median():.0f}"
)
print(
    f"median number of jobs per candidate: {df.group_by('CANDIDATE_ID').len()['len'].median():.0f}"
)
display(protocol_sizes(df, "group"))

kept by the protocol (jobs with at least 5 Female or Male candidates):
  pairs:      36,760 of 48,816 (75.3%)
  jobs:       3,344 of 9,180
  candidates: 6,510 of 7,517
median number of candidates per job: 9
median number of jobs per candidate: 6


group,pairs,jobs,candidates
str,u32,u32,u32
"""Female_CENTER""",1669,365,402
"""Female_NORTH""",13798,2394,2290
"""Female_SOUTH""",2022,326,397
"""Male_CENTER""",1631,386,360
"""Male_NORTH""",15346,2558,2580
"""Male_SOUTH""",2294,342,481
"""all""",36760,3344,6510


The filter keeps three quarters of the pairs: 36,760 pairs in 3,344 lists of at least 5 candidates (median 9), for
6,510 candidates with a median of 6 pairs each. The North holds most of the data (2,290 women and 2,580 men); the
other four groups have 360 to 481 candidates and 1,600 to 2,300 pairs each. A list mixes candidates of different
groups, so a selection is a competition between groups within the same job: the `jobs` column counts, for each group,
the lists in which it has at least one candidate.

### 2.2 The outcomes under the four scores

We rank the candidates of each job four times: with Adecco's score (`adecco`) and with our recomputed score on P, O and
C. Each row of the table is one of these rankings:

* `decided_by_draw`: share of pairs whose outcome is decided by the random draw, because they have the same score as
  another candidate of the job and the two sit on the two sides of the cutoff;
* `agree_adecco`: share of pairs that get the same outcome (selected or not) as with Adecco's score.

The first table uses `top_half` as the outcome, the second `top3`. Below the tables: the share of selected pairs, and
the pairs whose outcome changes when the parser is corrected (O → C).

In [ ]:
from hiring_cv_bias.fairness.protocol import VERSIONS, outcome_summary

display(outcome_summary(df, VERSIONS, "top_half"))
display(outcome_summary(df, VERSIONS, "top3"))
print(
    f"selected pairs (top_half), the same for every score: {df['top_half_O'].mean():.1%}"
)
gain = (~df["top_half_O"] & df["top_half_C"]).sum()
loss = (df["top_half_O"] & ~df["top_half_C"]).sum()
print(
    f"pairs whose top_half outcome changes from O to C: {(gain + loss) / df.height:.1%} "
    f"({gain:,} enter the top half, {loss:,} leave it)"
)

version,decided_by_draw,agree_adecco
str,f64,f64
"""adecco""",0.2079,1.0000
"""P""",0.1828,0.9671
"""O""",0.0601,0.8153
"""C""",0.0566,0.8427


version,decided_by_draw,agree_adecco
str,f64,f64
"""adecco""",0.1735,1.0000
"""P""",0.1542,0.9733
"""O""",0.0414,0.8616
"""C""",0.0393,0.8792


selected pairs (top_half), the same for every score: 52.4%
pairs whose top_half outcome changes from O to C: 8.2% (1,514 enter the top half, 1,514 leave it)


What the tables say:

* **Many of Adecco's decisions are a tie.** With Adecco's score, 21% of the pairs are tied with another candidate
  exactly at the cutoff of the top half, so their outcome is decided by the draw. With P it is 18%, with O and C 6%.
* **52.4% of the pairs are selected, with every score.** The draw makes every job select exactly ⌈n/2⌉ candidates.
  It is above 50% because lists with an odd number of candidates round up (5 candidates → 3 selected).
* **Our score reproduces Adecco's decision.** P agrees with Adecco on 96.7% of the pairs (97.3% for `top3`). This
  validates the reconstruction: formula, rules and estimate together.
* **The parser changes the decision of one pair in five.** O agrees with Adecco on 81.5% of the pairs.
* **Correcting the parser changes 8.2% of the decisions.** 1,514 pairs enter the top half and 1,514 leave it: the
  number of places in a job is fixed, so when a candidate enters, another one leaves. C agrees with Adecco on 84.3% of
  the pairs, closer than O. Phase 3 asks who these pairs are: whether some groups enter or leave more often than
  others.